# PlayStation Game Success Report

This notebook is structured for presentation and final reporting. It maps directly to the proposal questions and produces cleaner summary tables and visuals for a report or class submission.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SRC_DIR = PROJECT_ROOT / 'src'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from playstation_success.data import load_clean_dataset
from playstation_success.features import build_feature_frame
from playstation_success.model import train_baseline_model

sns.set_theme(style='whitegrid', context='talk')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda value: f'{value:,.2f}')
palette = ['#003087', '#0057b8', '#0070d1', '#4fa3ff', '#a7d3ff']


In [ ]:
df = load_clean_dataset()
feature_df = build_feature_frame(df)

print('Rows:', len(df))
print('Columns:', len(df.columns))
df.head()

## Dataset Summary

In [ ]:
summary = pd.DataFrame({
    'metric': ['rows', 'columns', 'missing_playstation_score', 'missing_metacritic_score', 'missing_price'],
    'value': [
        len(df),
        len(df.columns),
        int(df['playstation_score'].isna().sum()),
        int(df['metacritic_score'].isna().sum()),
        int(df['highest_price'].isna().sum()),
    ]
})
summary

## Q1. What Factors Influence A PlayStation Game's Success?

Here success is represented by `playstation_score`.

In [ ]:
factor_frame = feature_df[[
    'highest_price',
    'metacritic_score',
    'metacritic_user_score',
    'playstation_score',
    'playstation_rating_count',
    'release_year'
]].corr(numeric_only=True)
factor_frame[['playstation_score']].sort_values('playstation_score', ascending=False)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(factor_frame, annot=True, cmap='Blues', fmt='.2f')
plt.title('Correlation Matrix For Success-Related Variables')
plt.tight_layout()
plt.show()

## Q2. Do Cheaper Games Perform Better Than Expensive Games?

In [ ]:
price_summary = (
    feature_df.groupby('price_bucket', dropna=False)
    .agg(
        game_count=('game_name', 'count'),
        avg_playstation_score=('playstation_score', 'mean'),
        avg_metacritic_score=('metacritic_score', 'mean'),
        avg_rating_count=('playstation_rating_count', 'mean')
    )
    .reset_index()
)
price_summary

In [ ]:
plot_frame = price_summary.dropna(subset=['price_bucket'])
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_frame, x='price_bucket', y='avg_playstation_score', palette=palette)
plt.title('Average PlayStation Score By Price Bucket')
plt.xlabel('Price Bucket')
plt.ylabel('Average PlayStation Score')
plt.tight_layout()
plt.show()

## Q3. Which Genres Have The Highest Player Satisfaction?

In [ ]:
genre_summary = (
    feature_df.groupby('genre_primary', dropna=False)
    .agg(
        game_count=('game_name', 'count'),
        avg_playstation_score=('playstation_score', 'mean'),
        avg_metacritic_user_score=('metacritic_user_score', 'mean')
    )
    .query('game_count >= 30')
    .sort_values('avg_playstation_score', ascending=False)
    .reset_index()
)
genre_summary.head(10)

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(data=genre_summary.head(10), y='genre_primary', x='avg_playstation_score', palette=palette)
plt.title('Top Genres By Average PlayStation Score')
plt.xlabel('Average PlayStation Score')
plt.ylabel('Genre')
plt.tight_layout()
plt.show()

## Q4. Does Release Timing Affect Success?

In [ ]:
timing_summary = (
    feature_df.groupby(['release_year', 'release_quarter'], dropna=False)
    .agg(
        game_count=('game_name', 'count'),
        avg_playstation_score=('playstation_score', 'mean')
    )
    .reset_index()
    .dropna(subset=['release_year', 'release_quarter'])
)
timing_summary.tail(12)

In [ ]:
recent_timing = timing_summary.query('release_year >= 2014')
pivot_timing = recent_timing.pivot(index='release_year', columns='release_quarter', values='avg_playstation_score')
pivot_timing = pivot_timing.astype(float)

plt.figure(figsize=(12, 7))
sns.heatmap(pivot_timing, cmap='Blues', annot=False)
plt.title('Average PlayStation Score By Release Year And Quarter')
plt.xlabel('Release Quarter')
plt.ylabel('Release Year')
plt.tight_layout()
plt.show()

## Q5. Do Platform Listings Perform Better Than Single-Platform Listings?

In [ ]:
platform_mode_summary = (
    feature_df.assign(listing_type=np.where(feature_df['is_multi_platform_listing'] == 1, 'Multi-platform listing', 'Single-platform listing'))
    .groupby('listing_type')
    .agg(
        game_count=('game_name', 'count'),
        avg_playstation_score=('playstation_score', 'mean'),
        avg_rating_count=('playstation_rating_count', 'mean')
    )
    .reset_index()
)
platform_mode_summary

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(data=platform_mode_summary, x='listing_type', y='avg_playstation_score', palette=palette[:2])
plt.title('Single vs Multi-Platform Listing Performance')
plt.xlabel('Listing Type')
plt.ylabel('Average PlayStation Score')
plt.tight_layout()
plt.show()

## Baseline Prediction Model

In [ ]:
training_result = train_baseline_model(df)
pd.DataFrame([training_result.metrics])

In [ ]:
training_result.feature_importance.head(12)

In [ ]:
top_features = training_result.feature_importance.head(12).sort_values('importance')

plt.figure(figsize=(10, 7))
plt.barh(top_features['feature'], top_features['importance'], color='#0057b8')
plt.title('Top Features In The Baseline Random Forest Model')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Final Report Notes

Use this section to write final conclusions once the charts above are reviewed.

- Success appears most related to review signals, especially user review metrics where available.
- Price, genre, and release timing can be compared directly using the summary tables above.
- The prediction model is a baseline, not a final answer. It is useful for showing which features carry signal and how well the current dataset supports prediction.
